# 🧹 Data Cleaning Pipeline
## ERP Sales Analytics - Shoebadoo E-Commerce Data

**Ziel:**
- Duplikate entfernen (Überprüfungsschritt)
- Fehlende Werte intelligent behandeln
- Fehlende Informationen mithilfe von NLP aus Produktbeschreibungen extrahieren
- Abgeleitete Felder berechnen (total_amount)- Datentypen validieren und korrigieren
- Bereinigte Datensätze zur Qualitätsvalidierung speichern


**Basierend auf den Ergebnissen aus:** `01_data_exploration.ipynb`
**Wichtige zu behandelnde Punkte:**
1. ⚠️ PRODUCTS.product_name: 19.2% NULL
2. ⚠️ PRODUCTS.category: 29.8% NULL
3. ⚠️ PRODUCTS.price: 9.2% NULL
4. ⚠️ PRODUCTS.brand: 33.2% NULL
5. ⚠️ SALES.total_amount: 9.1% NULL
6. ⚠️ RETURNS.refunded_amount: 9.2% NULL

## 1. Setup & Imports

In [1]:
# Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, isnan, count, sum as spark_sum, 
    regexp_extract, upper, trim, coalesce, lit,
    length, lower, regexp_replace
)
from pyspark.sql.types import *
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports erfolgreich!")

✅ Imports erfolgreich!


## 2. Spark Session erstellen
Erstellt eine Spark-Session und unterdrückt warnings, die hier eigentlich unnötig wären.

In [3]:
# Warnings zu unterdrücken:
import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger("py4j").setLevel(logging.ERROR)


# Spark Session erstellen
spark = SparkSession.builder \
    .appName("ERP-Sales-Data-Cleaning") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

print(f"✅ Spark Session erstellt!")
print(f"   Spark Version: {spark.version}")
print(f"   App Name: {spark.sparkContext.appName}")

✅ Spark Session erstellt!
   Spark Version: 3.5.0
   App Name: ERP-Sales-Data-Cleaning


## 3.1 Daten laden 

Wir laden die Daten, mit denen wir im ersten Notebook gearbeitet haben.

In [4]:
# Pfad-Konfiguration 
INPUT_PATH = "/app/data/cleaned/1_data_exploration"   # ← Output von Notebook 1
OUTPUT_PATH = "/app/data/cleaned/2_data_cleaning"     # ← Finale bereinigte Daten


print(f"📂 Lade Daten aus: {INPUT_PATH}")
print("-" * 80)

try:
    # Spark liest Parquet-Ordner (nicht Dateien!)
    customers_df = spark.read.parquet(f"{INPUT_PATH}/customers_clean.parquet")
    print(f"✅ customers_df: {customers_df.count():,} rows")
    
    products_df = spark.read.parquet(f"{INPUT_PATH}/products_clean.parquet")
    print(f"✅ products_df:  {products_df.count():,} rows")
    
    sales_df = spark.read.parquet(f"{INPUT_PATH}/sales_clean.parquet")
    print(f"✅ sales_df:     {sales_df.count():,} rows")
    
    returns_df = spark.read.parquet(f"{INPUT_PATH}/returns_clean.parquet")
    print(f"✅ returns_df:   {returns_df.count():,} rows")
    
    print("\n🎉 Alle Dateien erfolgreich geladen!")
    
except Exception as e:
    print(f"❌ Fehler: {e}")
    print("\n💡 Prüfe ob die Dateien existieren:")
    print(f"   ls -la {INPUT_PATH}")

📂 Lade Daten aus: /app/data/cleaned/1_data_exploration
--------------------------------------------------------------------------------
✅ customers_df: 8,000 rows
✅ products_df:  500 rows
✅ sales_df:     437,896 rows
✅ returns_df:   43,789 rows

🎉 Alle Dateien erfolgreich geladen!


## 4. Duplikate entfernen (Verification)

Obwohl wir aus der Explorativen Datenanalyse Report wissen, dass **keine Duplikate** vorhanden sind, führen wir dies als Best Practice durch.

In [5]:
print("🔍 Entferne Duplikate basierend auf Primary Keys...\n")

# Vor der Deduplizierung
customers_before = customers_df.count()
products_before = products_df.count()
sales_before = sales_df.count()
returns_before = returns_df.count()

# Duplikate entfernen basierend auf Primary Keys
customers_clean = customers_df.dropDuplicates(["customer_id"])
products_clean = products_df.dropDuplicates(["product_id"])
sales_clean = sales_df.dropDuplicates(["sale_id"])
returns_clean = returns_df.dropDuplicates(["return_id"])

# Nach der Deduplizierung
customers_after = customers_clean.count()
products_after = products_clean.count()
sales_after = sales_clean.count()
returns_after = returns_clean.count()

# Report
print(f"CUSTOMERS: {customers_before:,} → {customers_after:,} ({customers_before - customers_after} removed)")
print(f"PRODUCTS:  {products_before:,} → {products_after:,} ({products_before - products_after} removed)")
print(f"SALES:     {sales_before:,} → {sales_after:,} ({sales_before - sales_after} removed)")
print(f"RETURNS:   {returns_before:,} → {returns_after:,} ({returns_before - returns_after} removed)")

print("\n✅ Deduplizierung abgeschlossen!")

🔍 Entferne Duplikate basierend auf Primary Keys...



CUSTOMERS: 8,000 → 8,000 (0 removed)
PRODUCTS:  500 → 500 (0 removed)
SALES:     437,896 → 437,896 (0 removed)
RETURNS:   43,789 → 43,789 (0 removed)

✅ Deduplizierung abgeschlossen!


## 5. Missing Values - Initial Fills
Wir gehen hier Schritt für Schritt durch. Es gibt nämlich **verschiedene Arten von fehlenden Werten** in den Parquets.

Diese sind:
- NULL-Werte oder auch NONE in Python
- NaN-Werte oder auch Not a Number - nur bei Float/Double
- Zero also 0 oder 0.0

Wir füllen also erstmal **Text-Felder/Strings mit "UNKNOWN"** um diese zu flaggen und für spätere NLP Extraction.
Danach schauen wir, dass wir alle **numerischen Felder, die fehlen auf NULL / NONE** konvertieren und normalisieren, um damit später besser arbeiten zu können

### 5.1. Text-Felder mit 'UNKNOWN' füllen
- Füllt alle String-Felder mit 'UNKNOWN'
- Zeigt vorher/nachher Statistiken
- Lässt numerische Felder bewusst aus

In [6]:
print("🔧 Fülle fehlende Werte mit Platzhaltern...\n")

# CUSTOMERS (eigentlich keine NULLs vorhanden, aber als Best Practice defensive Programmierung)
customers_clean = customers_clean.fillna({
    'first_name': 'UNKNOWN',
    'last_name': 'UNKNOWN',
    'email': 'unknown@example.com',
    'country': 'UNKNOWN'
})
print("✅ CUSTOMERS: Platzhalter gesetzt")

# PRODUCTS (die Tabelle mit den meisten NULL-Werten!)
# Produktname 19.2%, category 29.8%, brand 33.2%
products_clean = products_clean.fillna({
    'product_name': 'UNKNOWN',
    'category': 'UNKNOWN',
    'brand': 'UNKNOWN',
    # price bleibt NULL - wird später berechnet/gefüllt!
})
print("✅ PRODUCTS: Platzhalter gesetzt (werden später durch NLP ersetzt)")

# SALES
sales_clean = sales_clean.fillna({
    'channel': 'UNKNOWN',
    'payment_method': 'UNKNOWN'
    # total_amount bleibt NULL/NaN - wird in Sektion 6 berechnet!
})
print("✅ SALES: Platzhalter gesetzt")

# RETURNS
returns_clean = returns_clean.fillna({
    'return_reason': 'UNKNOWN'
    # refunded_amount bleibt NULL - business logic entscheidet später
})
print("✅ RETURNS: Platzhalter gesetzt")

print("\n🎉 Initiale Platzhalter gesetzt!")

🔧 Fülle fehlende Werte mit Platzhaltern...

✅ CUSTOMERS: Platzhalter gesetzt
✅ PRODUCTS: Platzhalter gesetzt (werden später durch NLP ersetzt)
✅ SALES: Platzhalter gesetzt
✅ RETURNS: Platzhalter gesetzt

🎉 Initiale Platzhalter gesetzt!


### 5.1.5 Missing Values - überprüfen obs geklappt hat
Wir wissen vom vorherigen Report, dass:


1. ⚠️ PRODUCTS.product_name: 19.2% NULL
2. ⚠️ PRODUCTS.category: 29.8% NULL
3. ⚠️ PRODUCTS.price: 9.2% NULL
4. ⚠️ PRODUCTS.brand: 33.2% NULL
5. ⚠️ SALES.total_amount: 9.1% NULL
6. ⚠️ RETURNS.refunded_amount: 9.2% NULL


Deswegen überprüfen wir mal products, sales und returns

In [24]:
from pyspark.sql.functions import col

# Liste aller Spalten
columns = products_clean.columns

# Bedingung: irgendeine Spalte enthält 'UNKNOWN'
condition = None
for c in columns:
    cond_col = col(c).cast("string").contains("UNKNOWN")
    condition = cond_col if condition is None else (condition | cond_col)

# Filter anwenden und 5 Zeilen anzeigen
products_clean.filter(condition).show(5, truncate=False)


+----------+-------------------+--------+------+-------+---------------------------------------------------------------------------------------------------+
|product_id|product_name       |category|price |brand  |description                                                                                        |
+----------+-------------------+--------+------+-------+---------------------------------------------------------------------------------------------------+
|502       |UNKNOWN            |UNKNOWN |0.0   |UNKNOWN|Das beliebte Produkt der Marke Puma aus der Kategorie Sport jetzt für nur XX,XX Euro erhältlich!   |
|505       |UNKNOWN            |UNKNOWN |14.76 |UNKNOWN|Das beliebte Produkt der Marke Bench aus der Kategorie Zubehör jetzt für nur 14.76 Euro erhältlich!|
|509       |Boden Alte Spitze  |Sport   |273.82|UNKNOWN|Sein Vater Fenster einigen groß bald die Milch Fußball.                                            |
|512       |UNKNOWN            |UNKNOWN |163.6 |UNKNOWN|Da

### 5.3 Numerische Felder: NaN → NULL normalisieren 
- Konvertiert alle NaN zu NULL
- Konvertiert auch Zero (bei total_amount) zu NULL
- Einheitliche Behandlung aller fehlenden numerischen Werte

In [9]:
print("REPORT: NaN zu NULL konvertieren")
print("-" * 80)
print("⚠️  NaN-Werte sind problematisch und inkonsistent")
print("✅ Wir konvertieren alle NaN → NULL für einheitliche Behandlung")
print()

# ============================================================================
# PRODUCTS: price normalisieren
# ============================================================================
print("🔹 PRODUCTS: Normalisiere 'price'")

# Zähle verschiedene "fehlende" Typen VOR Normalisierung
price_null_before = products_clean.filter(col("price").isNull()).count()
price_nan_before = products_clean.filter(isnan(col("price"))).count()

print(f"   Vor Normalisierung:")
print(f"      NULL: {price_null_before:,}")
print(f"      NaN:  {price_nan_before:,}")

# Normalisiere: NaN → NULL
products_clean = products_clean.withColumn(
    "price",
    when(isnan(col("price")), lit(None))  # NaN wird zu NULL
    .otherwise(col("price"))  # Andere Werte bleiben
)

# Zähle NACH Normalisierung
price_null_after = products_clean.filter(col("price").isNull()).count()
price_nan_after = products_clean.filter(isnan(col("price"))).count()

print(f"   Nach Normalisierung:")
print(f"      NULL: {price_null_after:,}")
print(f"      NaN:  {price_nan_after:,}")
print(f"   ✅ {price_nan_before:,} NaN → NULL konvertiert")
print()

# ============================================================================
# SALES: total_amount normalisieren
# ============================================================================
print("🔹 SALES: Normalisiere 'total_amount'")

# Zähle verschiedene "fehlende" Typen VOR Normalisierung
total_null_before = sales_clean.filter(col("total_amount").isNull()).count()
total_nan_before = sales_clean.filter(isnan(col("total_amount"))).count()
total_zero_before = sales_clean.filter(col("total_amount") == 0).count()

print(f"   Vor Normalisierung:")
print(f"      NULL: {total_null_before:,}")
print(f"      NaN:  {total_nan_before:,}")
print(f"      Zero: {total_zero_before:,}")

# Normalisiere: NaN → NULL, Zero → NULL (da Zero = "fehlend" hier)
sales_clean = sales_clean.withColumn(
    "total_amount",
    when(isnan(col("total_amount")), lit(None))  # NaN → NULL
    .when(col("total_amount") == 0, lit(None))    # 0 → NULL (da fehlend)
    .otherwise(col("total_amount"))
)

# Zähle NACH Normalisierung
total_null_after = sales_clean.filter(col("total_amount").isNull()).count()
total_nan_after = sales_clean.filter(isnan(col("total_amount"))).count()

print(f"   Nach Normalisierung:")
print(f"      NULL: {total_null_after:,}")
print(f"      NaN:  {total_nan_after:,}")
print(f"   ✅ Alle NaN und Zero → NULL konvertiert")
print()

# ============================================================================
# RETURNS: refunded_amount normalisieren
# ============================================================================
print("🔹 RETURNS: Normalisiere 'refunded_amount'")

# Zähle VOR
refund_null_before = returns_clean.filter(col("refunded_amount").isNull()).count()
refund_nan_before = returns_clean.filter(isnan(col("refunded_amount"))).count()

print(f"   Vor Normalisierung:")
print(f"      NULL: {refund_null_before:,}")
print(f"      NaN:  {refund_nan_before:,}")

# Normalisiere
returns_clean = returns_clean.withColumn(
    "refunded_amount",
    when(isnan(col("refunded_amount")), lit(None))
    .otherwise(col("refunded_amount"))
)

# Zähle NACH
refund_null_after = returns_clean.filter(col("refunded_amount").isNull()).count()

print(f"   Nach Normalisierung:")
print(f"      NULL: {refund_null_after:,}")
print(f"   ✅ Normalisierung abgeschlossen")
print()

print("=" * 80)
print("✅ Numerische Felder normalisiert (alle NaN → NULL)!")
print("=" * 80)
print()

REPORT: NaN zu NULL konvertieren
--------------------------------------------------------------------------------
⚠️  NaN-Werte sind problematisch und inkonsistent
✅ Wir konvertieren alle NaN → NULL für einheitliche Behandlung

🔹 PRODUCTS: Normalisiere 'price'
   Vor Normalisierung:
      NULL: 46
      NaN:  0
   Nach Normalisierung:
      NULL: 46
      NaN:  0
   ✅ 0 NaN → NULL konvertiert

🔹 SALES: Normalisiere 'total_amount'
   Vor Normalisierung:
      NULL: 39,934
      NaN:  0
      Zero: 0
   Nach Normalisierung:
      NULL: 39,934
      NaN:  0
   ✅ Alle NaN und Zero → NULL konvertiert

🔹 RETURNS: Normalisiere 'refunded_amount'
   Vor Normalisierung:
      NULL: 4,008
      NaN:  0
   Nach Normalisierung:
      NULL: 4,008
   ✅ Normalisierung abgeschlossen

✅ Numerische Felder normalisiert (alle NaN → NULL)!



### 5.4 Final Verification
- Zeigt alle verbleibenden NULL-Werte
- Klare Übersicht pro Tabelle
- Dokumentiert was noch zu tun ist

In [10]:
print("🔍 Schritt 3: Final Verification - Übersicht fehlender Werte")
print("=" * 80)

def count_missing_values(df, table_name):
    """Zählt NULL-Werte für jede Spalte"""
    print(f"\n📊 {table_name}:")
    print("-" * 80)
    
    has_nulls = False
    for col_name in df.columns:
        null_count = df.filter(col(col_name).isNull()).count()
        if null_count > 0:
            percentage = (null_count / df.count()) * 100
            print(f"   {col_name:20} {null_count:>8,} NULL ({percentage:>5.1f}%)")
            has_nulls = True
    
    if not has_nulls:
        print("   ✅ Keine NULL-Werte!")
    
    return has_nulls

# Prüfe alle Tabellen
count_missing_values(customers_clean, "CUSTOMERS")
count_missing_values(products_clean, "PRODUCTS")
count_missing_values(sales_clean, "SALES")
count_missing_values(returns_clean, "RETURNS")

print("\n" + "=" * 80)
print("✅ SEKTION 5 ABGESCHLOSSEN: Initial Fill & Normalisierung")
print("=" * 80)
print()
print("📝 Zusammenfassung:")
print("   ✅ Text-Felder mit 'UNKNOWN' gefüllt (für NLP-Extraktion)")
print("   ✅ Numerische Felder auf NULL normalisiert (NaN → NULL)")
print("   ✅ Verbleibende NULLs sind GEWOLLT für spätere Berechnungen")
print()
print("➡️  Nächste Schritte:")
print("   📌 Sektion 6: total_amount berechnen")
print("   📌 Sektion 7: NLP-Extraktion für 'UNKNOWN' Werte")
print()

🔍 Schritt 3: Final Verification - Übersicht fehlender Werte

📊 CUSTOMERS:
--------------------------------------------------------------------------------
   ✅ Keine NULL-Werte!

📊 PRODUCTS:
--------------------------------------------------------------------------------
   price                      46 NULL (  9.2%)

📊 SALES:
--------------------------------------------------------------------------------
   total_amount           39,934 NULL (  9.1%)

📊 RETURNS:
--------------------------------------------------------------------------------
   refunded_amount         4,008 NULL (  9.2%)

✅ SEKTION 5 ABGESCHLOSSEN: Initial Fill & Normalisierung

📝 Zusammenfassung:
   ✅ Text-Felder mit 'UNKNOWN' gefüllt (für NLP-Extraktion)
   ✅ Numerische Felder auf NULL normalisiert (NaN → NULL)
   ✅ Verbleibende NULLs sind GEWOLLT für spätere Berechnungen

➡️  Nächste Schritte:
   📌 Sektion 6: total_amount berechnen
   📌 Sektion 7: NLP-Extraktion für 'UNKNOWN' Werte



## 6. Calculate Missing total_amount (SALES)

Fehlende `total_amount` Werte können berechnet werden: `price * quantity`
bei Sales fehlen die total_amount Werte, diese können mit Preis * Menge berechnet werden mit Join zu den products
join mit products um Preis zu bekommen

In [11]:
print("💰 Berechne fehlende total_amount Werte...\n")
print("=" * 80)

from pyspark.sql.functions import isnan, col, when

# ============================================================================
# SCHRITT 1: Count missing BEFORE
# ============================================================================
print("📊 Schritt 1: Zähle fehlende Werte VOR Berechnung")
print("-" * 80)

# Nur NULL zählen (nicht 0!)
missing_before = sales_clean.filter(col("total_amount").isNull()).count()
print(f"NULL total_amount Werte: {missing_before:,}\n")

# ============================================================================
# SCHRITT 2: Join mit products um Preis zu bekommen
# ============================================================================
print("📊 Schritt 2: Join mit products für Preis")
print("-" * 80)

sales_clean = sales_clean.join(
    products_clean.select("product_id", col("price").alias("product_price")),
    on="product_id",
    how="left"
)

# Prüfe wie viele Produkte KEINEN Preis haben
null_price_count = sales_clean.filter(col("product_price").isNull()).count()
print(f"Sales-Rows mit NULL product_price: {null_price_count:,}")

if null_price_count > 0:
    print(f"⚠️  {null_price_count:,} sales haben kein Preis-Match")
    print("   (Diese bleiben 0.0 nach Berechnung)\n")
else:
    print("✅ Alle Produkte haben einen Preis!\n")

# ============================================================================
# SCHRITT 3: Berechne total_amount
# ============================================================================
print("📊 Schritt 3: Berechne total_amount = product_price * quantity")
print("-" * 80)

sales_clean = sales_clean.withColumn(
    "total_amount",
    when(
        # NUR wenn total_amount NULL ist (nicht 0!)
        col("total_amount").isNull(),
        # Berechne WENN product_price verfügbar
        when(
            col("product_price").isNotNull() & (col("product_price") > 0),
            col("product_price") * col("quantity")
        ).otherwise(lit(None))  # ← Bleibt NULL statt 0!
    ).otherwise(col("total_amount"))
)

# Cleanup
sales_clean = sales_clean.drop("product_price")

# ============================================================================
# SCHRITT 4: Count AFTER
# ============================================================================
print("\n📊 Schritt 4: Zähle fehlende Werte NACH Berechnung")
print("-" * 80)

missing_after = sales_clean.filter(col("total_amount").isNull()).count()
calculated = missing_before - missing_after

print(f"NULL total_amount Werte: {missing_after:,}")
print(f"✅ {calculated:,} Werte erfolgreich berechnet!")

# Zeige Verteilung
null_final = sales_clean.filter(col("total_amount").isNull()).count()
zero_final = sales_clean.filter(col("total_amount") == 0).count()
positive_final = sales_clean.filter(col("total_amount") > 0).count()

print(f"\n📈 Finale Verteilung:")
print(f"   NULL:     {null_final:,} ({null_final/sales_clean.count()*100:.1f}%)")
print(f"   Zero:     {zero_final:,} ({zero_final/sales_clean.count()*100:.1f}%)")
print(f"   Positive: {positive_final:,} ({positive_final/sales_clean.count()*100:.1f}%)")

# ============================================================================
# SCHRITT 5: Beispiele zeigen
# ============================================================================
print("\n📊 Beispiele BERECHNETER Werte:")
print("-" * 80)
sales_clean.filter(col("total_amount") > 0) \
    .select("sale_id", "product_id", "quantity", "total_amount") \
    .show(10, truncate=False)

# Zeige auch die NICHT berechneten (falls vorhanden)
if missing_after > 0:
    print(f"\n⚠️  Beispiele NICHT BERECHENBARER Werte (product_price = NULL):")
    print("-" * 80)
    # Rejoin kurz um zu zeigen warum
    sales_with_price = sales_clean.join(
        products_clean.select("product_id", "price"),
        on="product_id",
        how="left"
    )
    sales_with_price.filter(col("total_amount").isNull()) \
        .select("sale_id", "product_id", "quantity", "price", "total_amount") \
        .show(5)

print("\n" + "=" * 80)
print("✅ SEKTION 6 ABGESCHLOSSEN: total_amount Berechnung")
print("=" * 80)

💰 Berechne fehlende total_amount Werte...

📊 Schritt 1: Zähle fehlende Werte VOR Berechnung
--------------------------------------------------------------------------------
NULL total_amount Werte: 39,934

📊 Schritt 2: Join mit products für Preis
--------------------------------------------------------------------------------
Sales-Rows mit NULL product_price: 39,934
⚠️  39,934 sales haben kein Preis-Match
   (Diese bleiben 0.0 nach Berechnung)

📊 Schritt 3: Berechne total_amount = product_price * quantity
--------------------------------------------------------------------------------

📊 Schritt 4: Zähle fehlende Werte NACH Berechnung
--------------------------------------------------------------------------------
NULL total_amount Werte: 39,934
✅ 0 Werte erfolgreich berechnet!

📈 Finale Verteilung:
   NULL:     39,934 (9.1%)
   Zero:     0 (0.0%)
   Positive: 397,962 (90.9%)

📊 Beispiele BERECHNETER Werte:
------------------------------------------------------------------------------

## 7. NLP: Extract Category, Brand from Description
Hier wollen wir fehlende Strings bzw. Worte aus der Beschreibung extrahieren.
Wir gehen in dieser Sektion in drei Schritten vor und zwar einmal mit einer Wordlist und dann mit einem Regex-Pattern:
1. Wordlist-Match (schnell & präzise)
   ↓ Findet: ~70-80%
2. Regex-Pattern (flexibel)
   ↓ Findet: ~10-20% mehr
3. UNKNOWN bleibt (für manuelle Review)
   ↓ Rest: ~10%

Dabei machen wir dies in folgender Reihenfolge für die Worte
1. Brand extrahieren (Wordlist + Regex)
2. Category extrahieren (Wordlist + Regex)
3. Product_name extrahieren (einfacher)
SCHRITT 1: Wir extrahieren ECHTE Werte aus den Daten (Distinct), nehmen aus vorhandenden Daten schonmal eine Liste von Marken und Kategorien

In [9]:
print("📊 Schritt 1: Extrahiere bekannte Werte aus vorhandenen Daten...\n")

# Bekannte Kategorien - alle nicht-NULL, nicht-UNKNOWN Werte
known_categories_df = products_clean \
    .filter((col("category").isNotNull()) & (col("category") != "UNKNOWN")) \
    .select("category") \
    .distinct()

known_categories = [row.category for row in known_categories_df.collect()]
print(f"✅ Bekannte Kategorien ({len(known_categories)}): {known_categories}")

# Bekannte Marken - alle nicht-NULL, nicht-UNKNOWN Werte  
known_brands_df = products_clean \
    .filter((col("brand").isNotNull()) & (col("brand") != "UNKNOWN")) \
    .select("brand") \
    .distinct()

known_brands = [row.brand for row in known_brands_df.collect()]
print(f"✅ Bekannte Marken ({len(known_brands)}): {known_brands}")

# Falls die Listen leer sind (alle waren NULL), füge Fallback-Werte hinzu
if not known_categories:
    known_categories = ['Sport', 'Schuhe', 'Bekleidung', 'Accessoires']
    print(f"⚠️ Keine Kategorien gefunden - verwende Fallback: {known_categories}")

if not known_brands:
    known_brands = ['Boss', 'Eastpak', 'Nike', 'Adidas', 'Puma', 'Reebok']
    print(f"⚠️ Keine Marken gefunden - verwende Fallback: {known_brands}")

📊 Schritt 1: Extrahiere bekannte Werte aus vorhandenen Daten...

✅ Bekannte Kategorien (5): ['Accessoire', 'Schuhe', 'Zubehör', 'Kleidung', 'Sport']
✅ Bekannte Marken (9): ['Levis', 'Nike', 'Bench', 'Puma', 'Addidas', 'Esprit', 'Eastpak', 'Boss', 'FILA']


### 7.1 Extract Category from Description

In [ ]:
print("\n📦 Extrahiere Kategorie aus description...")

# Count UNKNOWN before
unknown_before = products_clean.filter(col("category") == "UNKNOWN").count()
print(f"UNKNOWN categories vorher: {unknown_before:,}")

# Extract category from description using regex
products_clean = products_clean.withColumn(
    "extracted_category",
    regexp_extract(col("description"), category_pattern, 1)
)

# Update category only where it was UNKNOWN and extraction found something
products_clean = products_clean.withColumn(
    "category",
    when(
        (col("category") == "UNKNOWN") & (length(col("extracted_category")) > 0),
        col("extracted_category")
    ).otherwise(col("category"))
)

# Drop temporary column
products_clean = products_clean.drop("extracted_category")

# Count UNKNOWN after
unknown_after = products_clean.filter(col("category") == "UNKNOWN").count()
print(f"UNKNOWN categories nachher: {unknown_after:,}")
print(f"✅ {unknown_before - unknown_after:,} Kategorien extrahiert!")

### 7.2 Extract Brand from Description

In [ ]:
print("\n🏷️ Extrahiere Marke aus description...")

# Count UNKNOWN before
unknown_before = products_clean.filter(col("brand") == "UNKNOWN").count()
print(f"UNKNOWN brands vorher: {unknown_before:,}")

# Create regex pattern: match any known brand (case-insensitive)
brand_pattern = r'\b(' + '|'.join(known_brands) + r')\b'

# Extract brand from description
products_clean = products_clean.withColumn(
    "extracted_brand",
    regexp_extract(upper(col("description")), brand_pattern.upper(), 1)
)

# Update brand only where it was UNKNOWN and extraction found something
products_clean = products_clean.withColumn(
    "brand",
    when(
        (col("brand") == "UNKNOWN") & (length(col("extracted_brand")) > 0),
        col("extracted_brand")
    ).otherwise(col("brand"))
)

# Drop temporary column
products_clean = products_clean.drop("extracted_brand")

# Count UNKNOWN after
unknown_after = products_clean.filter(col("brand") == "UNKNOWN").count()
print(f"UNKNOWN brands nachher: {unknown_after:,}")
print(f"✅ {unknown_before - unknown_after:,} Marken extrahiert!")

### 7.3 Extract Product Name (first N words from description)

In [ ]:
print("\n📝 Extrahiere Produktname aus description...")

# Count UNKNOWN before
unknown_before = products_clean.filter(col("product_name") == "UNKNOWN").count()
print(f"UNKNOWN product_names vorher: {unknown_before:,}")

# Extract first 3-5 words from description as product name
# Regex: Match first 3-5 words (max 50 chars)
name_pattern = r'^([A-Za-zäöüÄÖÜß0-9\s]{1,50})'

products_clean = products_clean.withColumn(
    "extracted_name",
    trim(regexp_extract(col("description"), name_pattern, 1))
)

# Update product_name only where it was UNKNOWN
products_clean = products_clean.withColumn(
    "product_name",
    when(
        (col("product_name") == "UNKNOWN") & (length(col("extracted_name")) > 0),
        col("extracted_name")
    ).otherwise(col("product_name"))
)

# Drop temporary column
products_clean = products_clean.drop("extracted_name")

# Count UNKNOWN after
unknown_after = products_clean.filter(col("product_name") == "UNKNOWN").count()
print(f"UNKNOWN product_names nachher: {unknown_after:,}")
print(f"✅ {unknown_before - unknown_after:,} Produktnamen extrahiert!")

print("\n🎉 NLP-Extraktion abgeschlossen!")

## 8. Data Type Validation & Correction

Stelle sicher, dass alle Spalten die korrekten Datentypen haben.

In [ ]:
print("🔧 Validiere und korrigiere Datentypen...\n")

# PRODUCTS: Preis muss positiv sein
negative_prices = products_clean.filter(col("price") < 0).count()
print(f"⚠️ Negative Preise gefunden: {negative_prices}")

if negative_prices > 0:
    # Setze negative Preise auf 0 (oder absoluten Wert)
    products_clean = products_clean.withColumn(
        "price",
        when(col("price") < 0, 0).otherwise(col("price"))
    )
    print("✅ Negative Preise auf 0 gesetzt")

# SALES: Menge muss positiv sein
negative_qty = sales_clean.filter(col("quantity") <= 0).count()
print(f"⚠️ Ungültige Mengen gefunden: {negative_qty}")

if negative_qty > 0:
    # Setze ungültige Mengen auf 1
    sales_clean = sales_clean.withColumn(
        "quantity",
        when(col("quantity") <= 0, 1).otherwise(col("quantity"))
    )
    print("✅ Ungültige Mengen auf 1 gesetzt")

# SALES: total_amount muss >= 0 sein
negative_amounts = sales_clean.filter(col("total_amount") < 0).count()
print(f"⚠️ Negative Beträge gefunden: {negative_amounts}")

if negative_amounts > 0:
    sales_clean = sales_clean.withColumn(
        "total_amount",
        when(col("total_amount") < 0, 0).otherwise(col("total_amount"))
    )
    print("✅ Negative Beträge auf 0 gesetzt")

# String Spalten trimmen (remove leading/trailing spaces)
string_columns = [field.name for field in products_clean.schema.fields if isinstance(field.dataType, StringType)]
for col_name in string_columns:
    products_clean = products_clean.withColumn(col_name, trim(col(col_name)))

print("\n✅ Datentyp-Validierung abgeschlossen!")

## 9. Final Quality Check

Prüfe die Datenqualität nach dem Cleaning.

In [ ]:
print("\n" + "="*80)
print(" 📊 FINAL QUALITY CHECK - AFTER CLEANING")
print("="*80 + "\n")

def quick_quality_check(df, name):
    print(f"📦 {name}:")
    total = df.count()
    print(f"   Total Rows: {total:,}")
    
    # Count UNKNOWN values
    for col_name in df.columns:
        unknown_count = df.filter(col(col_name) == "UNKNOWN").count()
        if unknown_count > 0:
            percentage = unknown_count / total * 100
            print(f"   ⚠️ {col_name}: {unknown_count:,} UNKNOWN ({percentage:.1f}%)")
    print()

quick_quality_check(customers_clean, "CUSTOMERS")
quick_quality_check(products_clean, "PRODUCTS")
quick_quality_check(sales_clean, "SALES")
quick_quality_check(returns_clean, "RETURNS")

print("="*80)

## 10. Sample Preview - Verify Cleaning Results

In [ ]:
print("\n👀 PRODUCTS Sample (nach Cleaning):")
products_clean.select("product_id", "product_name", "category", "brand", "price").show(5, truncate=False)

## 11. Save Cleaned Data

Speichere die bereinigten Daten als Parquet über pandas für die nächste Phase.

In [ ]:


OUTPUT_PATH = "/app/data/cleaned"

print(f"\n💾 Speichere BEREINIGTE Daten nach: {OUTPUT_PATH}")
print("-" * 80)
print("🔧 Verwende Pandas zum Speichern (vermeidet Spark-Probleme)")
print()

try:
    # Konvertiere Spark → Pandas und speichere
    print("📦 Speichere CUSTOMERS...")
    customers_clean.toPandas().to_parquet(
        f"{OUTPUT_PATH}/customers_clean.parquet",
        engine='pyarrow',
        compression='snappy'
    )
    print("✅ customers_clean.parquet gespeichert")
    
    print("📦 Speichere PRODUCTS...")
    products_clean.toPandas().to_parquet(
        f"{OUTPUT_PATH}/products_clean.parquet",
        engine='pyarrow',
        compression='snappy'
    )
    print("✅ products_clean.parquet gespeichert")
    
    print("📦 Speichere RETURNS...")
    returns_clean.toPandas().to_parquet(
        f"{OUTPUT_PATH}/returns_clean.parquet",
        engine='pyarrow',
        compression='snappy'
    )
    print("✅ returns_clean.parquet gespeichert")
    
    # SALES ist groß - kann dauern
    print("📦 Speichere SALES (große Datei - kann 30-60 Sekunden dauern)...")
    sales_clean.toPandas().to_parquet(
        f"{OUTPUT_PATH}/sales_clean.parquet",
        engine='pyarrow',
        compression='snappy'
    )
    print("✅ sales_clean.parquet gespeichert")
    
    print("\n🎉 Alle BEREINIGTEN Dateien erfolgreich gespeichert!")
    print("📝 Format: Pandas Parquet (single file)")
    print("✅ Bereit für Notebook 3: Quality Validation!")
    
except MemoryError:
    print("⚠️ Nicht genug RAM für große Dateien!")
    print("💡 Fallback: Speichere als CSV...")
    
    # CSV Fallback
    sales_clean.toPandas().to_csv(f"{OUTPUT_PATH}/sales_clean.csv", index=False)
    print("✅ sales_clean.csv gespeichert (CSV Fallback)")
    
except Exception as e:
    print(f"❌ Fehler beim Speichern: {e}")
```


## 12. Cleaning Summary Report

In [ ]:
print("\n" + "="*80)
print(" 📋 DATA CLEANING SUMMARY")
print("="*80 + "\n")

print("✅ **Completed Tasks:**")
print("   1. Removed duplicates (verification)")
print("   2. Filled missing values with intelligent defaults")
print("   3. Calculated missing total_amount in SALES")
print("   4. Extracted category from product descriptions")
print("   5. Extracted brand from product descriptions")
print("   6. Extracted product names from descriptions")
print("   7. Validated and corrected data types")
print("   8. Saved cleaned datasets")

print("\n📊 **Quality Improvements:**")
print("   - PRODUCTS: Reduced NULL values significantly")
print("   - SALES: Calculated all missing amounts")
print("   - All tables: Validated data types and ranges")

print("\n⏭️ **Next Steps:**")
print("   - Run 03_data_quality_validation.ipynb")
print("   - Implement Great Expectations validation")
print("   - Define data contracts")
print("   - Proceed to dimensional modeling")

print("\n" + "="*80)

In [ ]:
# Spark Session beenden
spark.stop()
print("\n✅ Spark Session beendet. Data Cleaning abgeschlossen!")